# AI4Mars MSL NavCam segmentation

Runtime → Change runtime type → **T4 GPU**.

Runtime → Change runtime type → **T4 GPU**.

`SMOKE = False` trains the full set for 10 epochs. If Colab drops, re-run from the mount cell; training resumes from `last.pt`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import sys
from pathlib import Path

import torch

ROOT = Path("/content/drive/MyDrive/mars-terrain")
sys.path.insert(0, str(ROOT))

SMOKE = False
print("gpu", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("root", ROOT, "exists", ROOT.is_dir())
print("src", (ROOT / "src").is_dir())

In [ ]:
from src.prepare import ensure_local_ncam

NCAM = ensure_local_ncam(ROOT, Path("/content/data"))
print(NCAM)
print("images", len(list((NCAM / "images" / "edr").glob("*.JPG"))))
print("train labels", len(list((NCAM / "labels" / "train").glob("*.png"))))

In [ ]:
from src.config import TrainConfig
from src.train import train

cfg = TrainConfig(
    image_size=512,
    batch_size=4 if SMOKE else 8,
    epochs=1 if SMOKE else 10,
    max_train=128 if SMOKE else None,
    num_workers=2,
    resume=True,
)
best = train(NCAM, cfg, project=ROOT)
print(best)

In [ ]:
from src.dataset import load_index, make_loader
from src.evaluate import evaluate, format_metrics
from src.model import build_model
from src.paths import output_dir

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
out = output_dir(ROOT)
ckpt = torch.load(out / "checkpoints" / "best.pt", map_location=device)
model = build_model().to(device)
model.load_state_dict(ckpt["model"])

val_pairs = load_index(NCAM, "min1", Path("/content/cache"), skip_empty=True)
val_loader = make_loader(val_pairs, 512, 8, False, 2, False)
metrics = evaluate(model, val_loader, device)
print(format_metrics(metrics))
print("checkpoint epoch", ckpt.get("epoch"))

In [ ]:
from src.preview import save_previews

paths = save_previews(NCAM, ROOT, n=8)
print("saved", len(paths), "to", paths[0].parent)

In [ ]:
import matplotlib.pyplot as plt
from src.visualize import denormalize, overlay

model.eval()
batch = next(iter(val_loader))
images = batch["image"][:4].to(device)
labels = batch["label"][:4].numpy()
with torch.no_grad():
    preds = model(images)["out"].argmax(1).cpu().numpy()

fig, axes = plt.subplots(4, 3, figsize=(10, 12))
for i in range(4):
    rgb = denormalize(images[i])
    axes[i, 0].imshow(rgb)
    axes[i, 1].imshow(overlay(rgb, labels[i]))
    axes[i, 2].imshow(overlay(rgb, preds[i]))
    for ax, title in zip(axes[i], ("image", "expert", "pred")):
        ax.set_title(title)
        ax.axis("off")
plt.tight_layout()
plt.show()